# 2: Exploratory Data Analysis

Goal: Profile the merged dataset and surface the descriptive patterns worth testing formally. 

**Created files:**

- `outputs/figures/eda_01_delay_minutes_distribution.png`: overall delay distribution and target balance figure.
- `outputs/figures/eda_02_delay_rate_by_season_window.png` through `outputs/figures/eda_10_delay_rate_by_operator.png`: temporal and operational delay-rate figures.
- `outputs/figures/eda_11_weather_hourly_distributions.png`: weather sample overview figure.
- `outputs/figures/eda_12_delay_rate_by_temperature_bin.png` through `outputs/figures/eda_15_delay_rate_by_wind_speed_bin.png`: grouped weather-condition delay-rate figures.
- `outputs/figures/eda_16_hourly_weather_vs_delay_rate.png`: continuous weather versus hourly delay-rate figure.
- `outputs/figures/eda_17_numeric_correlation_heatmap.png`, `outputs/figures/eda_18_hour_weekday_delay_heatmap.png`, and `outputs/figures/eda_19_rain_effect_by_hour_and_line.png`: multivariate interaction figures used to summarize combined patterns.


## 2.1 Setup and load the merged dataset

This section loads the merged dataset and prepares the figure output folder so the rest of the notebook can focus on analysis rather than file setup.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.ticker import PercentFormatter

NOTEBOOK_DIR = Path.cwd().resolve()
# Support running the notebook either inside notebooks/ or from the project root.
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
# EDA always starts from the slim merged dataset created in data_collection.ipynb.
DATASET_PATH = ROOT / "data" / "processed" / "dataset_final.csv"
OUTPUTS_DIR = ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# A white grid theme keeps percentage plots readable without overwhelming the report figures.
sns.set_theme(style="whitegrid", context="talk")

print(f"Project root: {ROOT}")
print(f"Dataset path: {DATASET_PATH.relative_to(ROOT)}")
print(f"Figures directory: {FIGURES_DIR.relative_to(ROOT)}")


In [ ]:
# Parse the key timestamp columns up front so temporal grouping and plotting use real datetimes.
date_columns = ["service_date", "scheduled_arrival", "weather_time"]

assert DATASET_PATH.exists(), "Run data_collection.ipynb first so data/processed/dataset_final.csv exists."

df = pd.read_csv(DATASET_PATH, parse_dates=date_columns)
dataset_row_count = len(df)
dataset_column_count = len(df.columns)

season_order = [
    "Winter (Jan window)",
    "Spring (Apr window)",
    "Summer (Jul window)",
    "Autumn (Oct window)",
]
# Preserve the intended seasonal sequence instead of letting alphabetical sorting scramble the windows.
df["season_window"] = pd.Categorical(
    df["season_window"],
    categories=season_order,
    ordered=True,
)

# These checks confirm the merge notebook produced a fully labeled, analysis table with weather matched to every row.
assert df["season_window"].notna().all(), "Every row should map to a seasonal window."
assert df["weather_matched"].all(), "The merged dataset should have full weather coverage."

print(f"Loaded rows: {dataset_row_count:,}")
print(f"Loaded columns: {dataset_column_count:,}")
display(df.head())


## 2.2 Dataset overview and integrity checks

This section checks the basic shape of the dataset, confirms the key timestamps line up as expected, and looks for missing values before I start interpreting plots.

In [ ]:
scheduled_date = df["scheduled_arrival"].dt.strftime("%Y-%m-%d")
service_date = df["service_date"].dt.strftime("%Y-%m-%d")
# Count rows where scheduled arrival spills past midnight so the report can explain any mismatch in the service date.
overnight_spillover_rows = (scheduled_date != service_date).sum()

overview_df = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique train types",
            "stations in the Lausanne area",
            "unique lines",
            "unique operators",
            "service date start",
            "service date end",
            "scheduled-arrival start",
            "scheduled-arrival end",
            "weather coverage",
            "overnight spillover rows",
        ],
        "value": [
            dataset_row_count,
            dataset_column_count,
            df["train_type"].nunique(),
            df["stop_name"].nunique(),
            df["line_name"].nunique(),
            df["operator"].nunique(),
            df["service_date"].min().date().isoformat(),
            df["service_date"].max().date().isoformat(),
            df["scheduled_arrival"].min().isoformat(sep=" "),
            df["scheduled_arrival"].max().isoformat(sep=" "),
            f"{df['weather_matched'].mean():.2%}",
            int(overnight_spillover_rows),
        ],
    }
)

station_counts = (
    df["stop_name"]
    .value_counts()
    .rename_axis("stop_name")
    .reset_index(name="rows")
)

display(overview_df)
display(station_counts)


In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
    # Use assign so the percentage is computed from the grouped table without creating a separate temporary object.
    .assign(missing_pct=lambda x: x["missing_rows"] / len(df))
)
missing_summary = missing_summary[missing_summary["missing_rows"] > 0].sort_values(
    "missing_rows", ascending=False
)

day_gap = (
    df["scheduled_arrival"].dt.normalize() - df["service_date"].dt.normalize()
).dt.days

timestamp_checks = pd.DataFrame(
    {
        "check": [
            "Rows missing critical timestamps",
            "Rows missing weather timestamp",
            "Rows missing any hourly weather variable",
            "Rows with weather_matched = False",
            "Rows with scheduled arrival on service date",
            "Rows with scheduled arrival one day after service date",
            "Rows with any other service date gap",
        ],
        "value": [
            int(df[["service_date", "scheduled_arrival", "delay_minutes", "delayed"]].isna().any(axis=1).sum()),
            int(df["weather_time"].isna().sum()),
            # Run this check across the four weather columns to confirm the merge did not leave partial hourly data behind.
            int(df[["temperature_2m", "precipitation", "snowfall", "wind_speed_10m"]].isna().any(axis=1).sum()),
            int((~df["weather_matched"]).sum()),
            int((day_gap == 0).sum()),
            int((day_gap == 1).sum()),
            int((~day_gap.isin([0, 1])).sum()),
        ],
    }
)

if missing_summary.empty:
    print(
        "The slim analysis dataset has no missing values in the retained analysis columns. "
        "The overnight spillover rows reflect trains scheduled shortly after midnight, not corrupted timestamps."
    )
else:
    print(
        "The remaining missing values are limited to retained analysis fields that still need interpretation. "
        "The overnight spillover rows reflect trains scheduled shortly after midnight, not corrupted timestamps."
    )
display(missing_summary)
display(timestamp_checks)


## 2.3 Delay target balance and delay distribution

This section summarizes how common delayed arrivals are and shows how `delay_minutes` is distributed.

**Files produced in next step:** `outputs/figures/eda_01_delay_minutes_distribution.png`

In [ ]:
class_balance = (
    df["delayed"]
    # Sort the binary labels so 0/1 always display in the natural order of on time versus late.
    .value_counts()
    .sort_index()
    .rename_axis("delayed")
    .reset_index(name="rows")
)
class_balance["label"] = class_balance["delayed"].map({0: "On time or early", 1: "Late"})
class_balance["share"] = class_balance["rows"] / len(df)

season_summary = (
    df.groupby("season_window", observed=True)
    .agg(
        rows=("delayed", "size"),
        late_arrivals=("delayed", "sum"),
        delay_rate=("delayed", "mean"),
        mean_delay_minutes=("delay_minutes", "mean"),
        median_delay_minutes=("delay_minutes", "median"),
    )
    .reset_index()
)

overall_delay_summary = pd.DataFrame(
    {
        "metric": ["overall delay rate", "late arrivals", "on-time-or-early arrivals"],
        "value": [
            df["delayed"].mean(),
            int(df["delayed"].sum()),
            int((1 - df["delayed"]).sum()),
        ],
    }
)

display(overall_delay_summary)
display(class_balance[["delayed", "label", "rows", "share"]])
display(season_summary)


In [ ]:
# Use extreme, tail, quartile, and median cut points to summarize both punctual arrivals and large delay outliers.
delay_quantiles = df["delay_minutes"].quantile([0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.0])
delay_quantiles_df = delay_quantiles.rename_axis("quantile").reset_index(name="delay_minutes")

lower_clip = delay_quantiles.loc[0.01]
upper_clip = delay_quantiles.loc[0.99]
plot_values = df["delay_minutes"].clip(lower=lower_clip, upper=upper_clip)

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(plot_values, bins=80, color="#1f4e79", edgecolor="white", linewidth=0.4)
ax.axvline(0, color="#b22222", linestyle="--", linewidth=2, label="on-time threshold")
ax.set_title("Delay Minutes Distribution")
ax.set_xlabel("Delay minutes (clipped to 1st–99th percentile for readability)")
ax.set_ylabel("Arrival records")
ax.legend()
fig.tight_layout()

delay_plot_path = FIGURES_DIR / "eda_01_delay_minutes_distribution.png"
fig.savefig(delay_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to {delay_plot_path.relative_to(ROOT)}")
display(delay_quantiles_df)


## 2.4 Plotting helpers

This section defines the helper functions that keep the grouped delay plots consistent.

Implementation detail: safe to skim on a first read.

In [ ]:
# Use explicit weekday ordering so seaborn/bar plots do not fall back to alphabetical day names.
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
weekend_label_map = {False: "Weekday", True: "Weekend"}


def build_delay_summary(
    data: pd.DataFrame,
    column: str,
    order: list | None = None,
    label_map: dict | None = None,
    sort_desc: bool = False,
) -> pd.DataFrame:
    summary = (
        data.groupby(column, observed=False)
        .agg(
            rows=("delayed", "size"),
            delay_rate=("delayed", "mean"),
            mean_delay_minutes=("delay_minutes", "mean"),
        )
        .reset_index()
    )
    summary = summary[summary["rows"] > 0].copy()
    summary["category"] = summary[column].map(label_map) if label_map else summary[column]

    if order is not None:
        ordered_labels = [label_map.get(item, item) if label_map else item for item in order]
        summary["category"] = pd.Categorical(
            summary["category"],
            categories=ordered_labels,
            ordered=True,
        )
        summary = summary.sort_values("category")
    elif sort_desc:
        summary = summary.sort_values(["delay_rate", "rows"], ascending=[False, False])

    return summary.reset_index(drop=True)


def save_delay_rate_plot(
    summary: pd.DataFrame,
    title: str,
    filename: str,
    x_label: str,
    rotate: int = 0,
    figsize: tuple[float, float] = (12, 6),
    annotate_counts: bool = False,
) -> Path:
    fig, ax = plt.subplots(figsize=figsize)
    category_labels = summary["category"].astype(str).tolist()
    positions = list(range(len(summary)))
    bars = ax.bar(positions, summary["delay_rate"], color="#1f4e79", width=0.8)
    ax.set_xticks(positions)
    ax.set_xticklabels(category_labels)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Delay rate")
    ax.set_ylim(0, max(0.05, summary["delay_rate"].max() * 1.15))
    if rotate:
        plt.setp(ax.get_xticklabels(), rotation=rotate, ha="right")
    if annotate_counts and len(summary) <= 10:
        for patch, row in zip(bars, summary.itertuples(index=False)):
            ax.text(
                patch.get_x() + patch.get_width() / 2,
                patch.get_height(),
                f"n={row.rows:,}",
                ha="center",
                va="bottom",
                fontsize=9,
            )
    fig.tight_layout()
    output_path = FIGURES_DIR / filename
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.show()
    return output_path


## 2.5 Temporal and operational delay rate views

This section looks at how delays are distributed according to temporal and operational features:
- Season 
- Hour
- Day of the week
- Month
- Whether the day is a weekend
- Station
- Train type
- Line
- Operator

**Files produced in next step:** `outputs/figures/eda_02_delay_rate_by_season_window.png` through `outputs/figures/eda_10_delay_rate_by_operator.png`

In [ ]:
# Drive the repeated bar plots from one spec list so every temporal/operational view uses the same summary and save logic.
rate_plot_specs = [
    {
        "column": "season_window",
        "title": "Delay Rate by Seasonal Window",
        "filename": "eda_02_delay_rate_by_season_window.png",
        "x_label": "Seasonal window",
        "order": season_order,
        "label_map": None,
        # rotate controls readability of the horizontal axis labels, while figsize makes dense categories easier to read in the exported report figure.
        "rotate": 15,
        "figsize": (12, 6),
        "sort_desc": False,
    },
    {
        "column": "scheduled_arrival_hour",
        "title": "Delay Rate by Scheduled Arrival Hour",
        "filename": "eda_03_delay_rate_by_hour.png",
        "x_label": "Scheduled arrival hour",
        "order": list(range(24)),
        "label_map": None,
        "rotate": 0,
        "figsize": (14, 6),
        "sort_desc": False,
    },
    {
        "column": "scheduled_arrival_weekday_name",
        "title": "Delay Rate by Day of Week",
        "filename": "eda_04_delay_rate_by_weekday.png",
        "x_label": "Day of week",
        "order": weekday_order,
        "label_map": None,
        "rotate": 20,
        "figsize": (12, 6),
        "sort_desc": False,
    },
    {
        "column": "scheduled_arrival_month",
        "title": "Delay Rate by Calendar Month",
        "filename": "eda_05_delay_rate_by_month.png",
        "x_label": "Calendar month",
        "order": sorted(df["scheduled_arrival_month"].unique()),
        "label_map": None,
        "rotate": 0,
        "figsize": (10, 6),
        "sort_desc": False,
    },
    {
        "column": "is_weekend",
        "title": "Delay Rate by Weekend Indicator",
        "filename": "eda_06_delay_rate_by_weekend_indicator.png",
        "x_label": "Weekend indicator",
        "order": [False, True],
        "label_map": weekend_label_map,
        "rotate": 0,
        "figsize": (8, 6),
        "sort_desc": False,
    },
    {
        "column": "stop_name",
        "title": "Delay Rate by Station",
        "filename": "eda_07_delay_rate_by_station.png",
        "x_label": "Station",
        "order": None,
        "label_map": None,
        "rotate": 20,
        "figsize": (12, 6),
        "sort_desc": True,
    },
    {
        "column": "train_type",
        "title": "Delay Rate by Train Type",
        "filename": "eda_08_delay_rate_by_train_type.png",
        "x_label": "Train type",
        "order": None,
        "label_map": None,
        "rotate": 0,
        "figsize": (12, 6),
        "sort_desc": True,
    },
    {
        "column": "line_name",
        "title": "Delay Rate by Line Name",
        "filename": "eda_09_delay_rate_by_line_name.png",
        "x_label": "Line name",
        "order": None,
        "label_map": None,
        "rotate": 65,
        "figsize": (16, 8),
        "sort_desc": True,
    },
    {
        "column": "operator",
        "title": "Delay Rate by Operator",
        "filename": "eda_10_delay_rate_by_operator.png",
        "x_label": "Operator",
        "order": None,
        "label_map": None,
        "rotate": 0,
        "figsize": (8, 6),
        "sort_desc": True,
    },
]


In [ ]:
temporal_operational_summaries = {}
saved_rate_plots = []

for spec in rate_plot_specs:
    summary = build_delay_summary(
        df,
        column=spec["column"],
        order=spec["order"],
        label_map=spec["label_map"],
        sort_desc=spec["sort_desc"],
    )
    temporal_operational_summaries[spec["column"]] = summary
    display(summary)
    saved_rate_plots.append(
        save_delay_rate_plot(
            summary,
            title=spec["title"],
            filename=spec["filename"],
            x_label=spec["x_label"],
            rotate=spec["rotate"],
            figsize=spec["figsize"],
            annotate_counts=True,
        )
    )

print("Saved temporal/operational figures:")
for path in saved_rate_plots:
    print(f"- {path.relative_to(ROOT)}")


## 2.6 Hourly weather sample overview and distributions

This section steps back from delays for a moment and describes the weather sample itself.

**Files produced in next step:** `outputs/figures/eda_11_weather_hourly_distributions.png`

In [ ]:
weather_hourly = (
    # Aggregate to hourly weather granularity because temperature, rain, snow, and wind are shared by all arrivals in that hour.
    df.groupby("weather_time", observed=False)
    .agg(
        arrival_records=("delayed", "size"),
        delay_rate=("delayed", "mean"),
        mean_delay_minutes=("delay_minutes", "mean"),
        temperature_2m=("temperature_2m", "first"),
        precipitation=("precipitation", "first"),
        snowfall=("snowfall", "first"),
        wind_speed_10m=("wind_speed_10m", "first"),
    )
    .reset_index()
    .sort_values("weather_time")
)

weather_distribution_summary = (
    weather_hourly[["temperature_2m", "precipitation", "snowfall", "wind_speed_10m"]]
    .describe()
    .T.reset_index()
    .rename(columns={"index": "weather_variable"})
)

display(weather_distribution_summary)
print(f"Hourly weather observations with at least one matched arrival: {len(weather_hourly):,}")
print(f"Rainy hourly observations: {(weather_hourly['precipitation'] > 0).sum():,}")
print(f"Snowy hourly observations: {(weather_hourly['snowfall'] > 0).sum():,}")


In [ ]:
weather_distribution_specs = [
    {
        "column": "temperature_2m",
        "title": "Temperature Distribution Across Matched Hours",
        "xlabel": "Temperature (C)",
        "bins": 30,
        "color": "#1f4e79",
    },
    {
        "column": "precipitation",
        "title": "Precipitation Distribution Across Matched Hours",
        "xlabel": "Precipitation in prior hour (mm)",
        "bins": 30,
        "color": "#2e8b57",
    },
    {
        "column": "snowfall",
        "title": "Snowfall Distribution Across Matched Hours",
        "xlabel": "Snowfall in prior hour (cm)",
        "bins": 30,
        "color": "#708090",
    },
    {
        "column": "wind_speed_10m",
        "title": "Wind-Speed Distribution Across Matched Hours",
        "xlabel": "Wind speed at 10m (km/h)",
        "bins": 30,
        "color": "#c26d2c",
    },
]


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, spec in zip(axes.flat, weather_distribution_specs):
    ax.hist(
        weather_hourly[spec["column"]],
        bins=spec["bins"],
        color=spec["color"],
        edgecolor="white",
        linewidth=0.5,
    )
    ax.set_title(spec["title"])
    ax.set_xlabel(spec["xlabel"])
    ax.set_ylabel("Hourly observations")

fig.tight_layout()
weather_distribution_path = FIGURES_DIR / "eda_11_weather_hourly_distributions.png"
fig.savefig(weather_distribution_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to {weather_distribution_path.relative_to(ROOT)}")


## 2.7 Weather condition delay rate views

This section looks at how delays are distributed according to hourly weather features:
- Temperature
- Precipitation
- Snow
- Wind speed

For this purpose, the weather features are discretized.

**Files produced in next step:** `outputs/figures/eda_12_delay_rate_by_temperature_bin.png` through `outputs/figures/eda_15_delay_rate_by_wind_speed_bin.png`

In [ ]:
weather_delay_df = df.copy()

# These bins turn continuous weather readings into interpretable report categories for grouped delay rate plots.
temperature_bin_order = [
    "<= 0 C",
    "0 to 5 C",
    "5 to 10 C",
    "10 to 15 C",
    "15 to 20 C",
    "20 to 25 C",
    "> 25 C",
]
wind_bin_order = [
    "0 to 5 km/h",
    "5 to 10 km/h",
    "10 to 15 km/h",
    "15 to 20 km/h",
    "20 to 25 km/h",
    "> 25 km/h",
]

weather_delay_df["temperature_bin"] = pd.cut(
    weather_delay_df["temperature_2m"],
    # Use coarse 5-degree bands so each temperature bin keeps enough rows for stable delay rate comparisons.
    bins=[-float("inf"), 0, 5, 10, 15, 20, 25, float("inf")],
    labels=temperature_bin_order,
    include_lowest=True,
)
weather_delay_df["wind_speed_bin"] = pd.cut(
    weather_delay_df["wind_speed_10m"],
    # Use simple 5 km/h steps so the wind plot stays readable and comparable to the temperature bins.
    bins=[0, 5, 10, 15, 20, 25, float("inf")],
    labels=wind_bin_order,
    include_lowest=True,
)
weather_delay_df["precipitation_condition"] = pd.Categorical(
    weather_delay_df["precipitation"].gt(0).map({False: "Dry hour", True: "Rainy hour"}),
    categories=["Dry hour", "Rainy hour"],
    ordered=True,
)
weather_delay_df["snowfall_condition"] = pd.Categorical(
    weather_delay_df["snowfall"].gt(0).map({False: "No snow", True: "Snowy hour"}),
    categories=["No snow", "Snowy hour"],
    ordered=True,
)


In [ ]:
weather_delay_specs = [
    {
        "column": "temperature_bin",
        "title": "Delay Rate by Temperature Bin",
        "filename": "eda_12_delay_rate_by_temperature_bin.png",
        "x_label": "Temperature bin",
        "order": temperature_bin_order,
        "rotate": 20,
        "figsize": (13, 6),
    },
    {
        "column": "precipitation_condition",
        "title": "Delay Rate on Dry vs Rainy Hours",
        "filename": "eda_13_delay_rate_dry_vs_rainy.png",
        "x_label": "Precipitation condition",
        "order": ["Dry hour", "Rainy hour"],
        "rotate": 0,
        "figsize": (8, 6),
    },
    {
        "column": "snowfall_condition",
        "title": "Delay Rate on Snow vs No-Snow Hours",
        "filename": "eda_14_delay_rate_snow_vs_no_snow.png",
        "x_label": "Snowfall condition",
        "order": ["No snow", "Snowy hour"],
        "rotate": 0,
        "figsize": (8, 6),
    },
    {
        "column": "wind_speed_bin",
        "title": "Delay Rate by Wind-Speed Bin",
        "filename": "eda_15_delay_rate_by_wind_speed_bin.png",
        "x_label": "Wind-speed bin",
        "order": wind_bin_order,
        "rotate": 20,
        "figsize": (13, 6),
    },
]


In [ ]:
weather_condition_summaries = {}
saved_weather_condition_plots = []

for spec in weather_delay_specs:
    summary = build_delay_summary(
        weather_delay_df,
        column=spec["column"],
        order=spec["order"],
    )
    weather_condition_summaries[spec["column"]] = summary
    display(summary)
    saved_weather_condition_plots.append(
        save_delay_rate_plot(
            summary,
            title=spec["title"],
            filename=spec["filename"],
            x_label=spec["x_label"],
            rotate=spec["rotate"],
            figsize=spec["figsize"],
            annotate_counts=True,
        )
    )

print("Saved weather-condition figures:")
for path in saved_weather_condition_plots:
    print(f"- {path.relative_to(ROOT)}")


## 2.8 Continuous weather relationships

This section keeps the weather variables continuous and asks whether the delay rate moves gradually with rain, wind, snowfall, or temperature rather than only by category.

**Files produced in next step:** `outputs/figures/eda_16_hourly_weather_vs_delay_rate.png`

In [ ]:
# These are continuous weather variables, so this section uses scatter/regression views rather than the earlier categorical bar plot helper.
weather_relationship_specs = [
    {
        "column": "temperature_2m",
        "title": "Temperature vs Hourly Delay Rate",
        "xlabel": "Temperature (C)",
        "color": "#1f4e79",
    },
    {
        "column": "precipitation",
        "title": "Precipitation vs Hourly Delay Rate",
        "xlabel": "Precipitation in prior hour (mm)",
        "color": "#2e8b57",
    },
    {
        "column": "snowfall",
        "title": "Snowfall vs Hourly Delay Rate",
        "xlabel": "Snowfall in prior hour (cm)",
        "color": "#708090",
    },
    {
        "column": "wind_speed_10m",
        "title": "Wind Speed vs Hourly Delay Rate",
        "xlabel": "Wind speed at 10m (km/h)",
        "color": "#c26d2c",
    },
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, spec in zip(axes.flat, weather_relationship_specs):
    sns.scatterplot(
        data=weather_hourly,
        x=spec["column"],
        y="delay_rate",
        size="arrival_records",
        sizes=(20, 140),
        alpha=0.30,
        color=spec["color"],
        legend=False,
        ax=ax,
    )
    sns.regplot(
        data=weather_hourly,
        x=spec["column"],
        y="delay_rate",
        scatter=False,
        ci=None,
        color="#222222",
        line_kws={"linewidth": 2},
        ax=ax,
    )
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(spec["title"])
    ax.set_xlabel(spec["xlabel"])
    ax.set_ylabel("Hourly delay rate")

fig.tight_layout()
weather_relationship_path = FIGURES_DIR / "eda_16_hourly_weather_vs_delay_rate.png"
fig.savefig(weather_relationship_path, dpi=200, bbox_inches="tight")
plt.show()

hourly_weather_relationship_summary = (
    weather_hourly[
        [
            "delay_rate",
            "mean_delay_minutes",
            "temperature_2m",
            "precipitation",
            "snowfall",
            "wind_speed_10m",
            "arrival_records",
        ]
    ]
    .corr(numeric_only=True)
    [["delay_rate", "mean_delay_minutes"]]
)

display(hourly_weather_relationship_summary)
print(f"Saved figure to {weather_relationship_path.relative_to(ROOT)}")


## 2.9 Multivariate interaction views

This section adds a few compact multivariate views so I can see whether the patterns from the earlier plots still hold when two dimensions are shown together.

**Files produced in next step:** `outputs/figures/eda_17_numeric_correlation_heatmap.png`, `outputs/figures/eda_18_hour_weekday_delay_heatmap.png`, and `outputs/figures/eda_19_rain_effect_by_hour_and_line.png`

In [ ]:
# Restrict the correlation view to numeric analysis variables that speak directly to timing, weather, and the delay target.
numeric_eda_df = df[
    [
        "delay_minutes",
        "delayed",
        "scheduled_arrival_hour",
        "scheduled_arrival_weekday",
        "scheduled_arrival_month",
        "temperature_2m",
        "precipitation",
        "snowfall",
        "wind_speed_10m",
    ]
].copy()
numeric_eda_df["is_weekend"] = df["is_weekend"].astype(int)

correlation_matrix = numeric_eda_df.corr(numeric_only=True)
display(correlation_matrix.round(3))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=ax,
)
ax.set_title("Correlation Heatmap for Numeric EDA Variables")
fig.tight_layout()

correlation_path = FIGURES_DIR / "eda_17_numeric_correlation_heatmap.png"
fig.savefig(correlation_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to {correlation_path.relative_to(ROOT)}")


In [ ]:
hour_weekday_delay = (
    df.pivot_table(
        index="scheduled_arrival_weekday_name",
        columns="scheduled_arrival_hour",
        values="delayed",
        # mean turns the binary delayed flag into a delay rate for each cell for each weekday and hour.
        aggfunc="mean",
    )
    .reindex(weekday_order)
)

hour_weekday_counts = (
    df.pivot_table(
        index="scheduled_arrival_weekday_name",
        columns="scheduled_arrival_hour",
        values="delayed",
        # size preserves sample sizes so the heatmap can be interpreted alongside cell coverage.
        aggfunc="size",
    )
    .reindex(weekday_order)
    .fillna(0)
    .astype(int)
)

display(hour_weekday_delay.round(3))
display(hour_weekday_counts)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(
    hour_weekday_delay,
    cmap="YlOrRd",
    annot=True,
    # Annotations in percent format make the delay rate cells readable without an extra manual conversion step.
    fmt=".0%",
    linewidths=0.3,
    linecolor="white",
    cbar_kws={"label": "Delay rate"},
    ax=ax,
)
colorbar = ax.collections[0].colorbar
colorbar.ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("Delay Rate by Day of Week and Scheduled Arrival Hour")
ax.set_xlabel("Scheduled arrival hour")
ax.set_ylabel("Day of week")
fig.tight_layout()

hour_weekday_path = FIGURES_DIR / "eda_18_hour_weekday_delay_heatmap.png"
fig.savefig(hour_weekday_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to {hour_weekday_path.relative_to(ROOT)}")


In [ ]:
interaction_df = df.copy()
interaction_df["rain_condition"] = pd.Categorical(
    interaction_df["precipitation"].gt(0).map({False: "Dry hour", True: "Rainy hour"}),
    categories=["Dry hour", "Rainy hour"],
    ordered=True,
)


def build_rain_gap_table(
    data: pd.DataFrame,
    group_column: str,
    min_dry_rows: int,
    min_rainy_rows: int,
) -> pd.DataFrame:
    summary = (
        data.groupby([group_column, "rain_condition"], observed=False)
        .agg(rows=("delayed", "size"), delay_rate=("delayed", "mean"))
        .reset_index()
    )
    rates = summary.pivot(index=group_column, columns="rain_condition", values="delay_rate")
    counts = summary.pivot(index=group_column, columns="rain_condition", values="rows")
    effect = pd.DataFrame(
        {
            group_column: rates.index,
            "dry_delay_rate": rates["Dry hour"],
            "rainy_delay_rate": rates["Rainy hour"],
            "dry_rows": counts["Dry hour"],
            "rainy_rows": counts["Rainy hour"],
        }
    ).reset_index(drop=True)
    effect = effect[(effect["dry_rows"] >= min_dry_rows) & (effect["rainy_rows"] >= min_rainy_rows)].copy()
    effect["rain_delay_rate_gap"] = effect["rainy_delay_rate"] - effect["dry_delay_rate"]
    return effect


rain_hour_effect = build_rain_gap_table(
    interaction_df,
    group_column="scheduled_arrival_hour",
    min_dry_rows=150,
    min_rainy_rows=50,
)

display(rain_hour_effect.round(3))

rain_line_effect = build_rain_gap_table(
    interaction_df,
    group_column="line_name",
    min_dry_rows=500,
    min_rainy_rows=100,
)
rain_line_effect["total_rows"] = rain_line_effect["dry_rows"] + rain_line_effect["rainy_rows"]
# Keep only the busiest lines so the second panel stays legible and focuses on operationally important services.
rain_line_effect = rain_line_effect.sort_values("total_rows", ascending=False).head(12)

display(
    rain_line_effect[
        [
            "line_name",
            "dry_rows",
            "rainy_rows",
            "dry_delay_rate",
            "rainy_delay_rate",
            "rain_delay_rate_gap",
        ]
    ].round(3)
)

In [ ]:
# Use a layout with two panels as a lightweight facet grid: hour on the left and the busiest lines on the right.
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

hour_colors = rain_hour_effect["rain_delay_rate_gap"].apply(
    lambda value: "#2e8b57" if value >= 0 else "#b22222"
)
axes[0].bar(
    rain_hour_effect["scheduled_arrival_hour"].astype(str),
    rain_hour_effect["rain_delay_rate_gap"],
    color=hour_colors,
)
axes[0].axhline(0, color="#222222", linewidth=1)
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].set_title("Rainy vs Dry Delay-Rate Gap by Hour")
axes[0].set_xlabel("Scheduled arrival hour")
axes[0].set_ylabel("Rainy minus dry delay rate")

line_effect_plot = rain_line_effect.sort_values("rain_delay_rate_gap")
line_colors = line_effect_plot["rain_delay_rate_gap"].apply(
    lambda value: "#2e8b57" if value >= 0 else "#b22222"
)
axes[1].barh(
    line_effect_plot["line_name"],
    line_effect_plot["rain_delay_rate_gap"],
    color=line_colors,
)
axes[1].axvline(0, color="#222222", linewidth=1)
axes[1].xaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].set_title("Rainy vs Dry Delay-Rate Gap by Line")
axes[1].set_xlabel("Rainy minus dry delay rate")
axes[1].set_ylabel("Line name")

fig.tight_layout()
rain_interaction_path = FIGURES_DIR / "eda_19_rain_effect_by_hour_and_line.png"
fig.savefig(rain_interaction_path, dpi=200, bbox_inches="tight")
plt.show()

print("Hour-level comparison keeps hours with at least 150 dry rows and 50 rainy rows.")
print("Line-level comparison keeps lines with at least 500 dry rows and 100 rainy rows.")
print(f"Saved figure to {rain_interaction_path.relative_to(ROOT)}")

## 2.10 Key EDA Takeaways

This section closes the descriptive stage by summarizing the patterns that are most worth testing formally.

Key takeaways:
- Delay rates shift clearly by scheduled arrival hour, weekday or weekend status, train type, and line.
- Rainy hours show higher delay rates than dry hours in both the hour level and line level interaction views.
- Snow looks weaker and less consistent than rain in the descriptive plots.
- Temperature and wind show some movement in hourly delay rates, but those patterns look more subtle than the main service and timing effects.
- These are the patterns I carry forward into the formal hypothesis tests.